# Domain knowledge NLP 

This notebook follows [this](https://huggingface.co/learn/nlp-course/en/chapter7/3) HuggingFace tutorial. 

Using a chunk of data, I fine-tune an existing BERT model to have domain specific knowledge. 

In [2]:
import pandas as pd
import numpy as np
import math
import collections
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, default_data_collator, TrainingArguments, Trainer, pipeline
import torch
from datasets import Dataset

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set-up 

In [3]:
model_checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

In [4]:
distilbert_num_parameters = model.num_parameters() / 1000000
print(f"Number of parameters in DistilBERT: {distilbert_num_parameters:.2f}M")

Number of parameters in DistilBERT: 66.99M


This is the text which I'm going to look at the choice of MASK for. 

In [5]:
text = "This is an awful [MASK]."

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

inputs = tokenizer(text, return_tensors="pt")
token_logits = model(**inputs).logits

# Find the location of [MASK] in the input and extract its logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

# Pick the 5 [MASK] candidates with the highest logits
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(text.replace("[MASK]", tokenizer.decode([token])))

This is an awful sight.
This is an awful thing.
This is an awful idea.
This is an awful coincidence.
This is an awful mess.


## Load the data for fine-tuning. 

Here I load a chunk of comments on planning applications. 

In [7]:
# # Load data
train_df = pd.read_csv('../outputs/train_comments.csv')
test_df = pd.read_csv('../outputs/test_comments.csv')

# Convert DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [8]:
# randomly show two comments, from column 'comment_text'
print(train_df['comment_text'][0])

The new development will increase noise pollution in this area


In [22]:
len(train_df['comment_text'])

542

## Tokenize the data 

This step tokenizes the data used for training. 

In [9]:
# Tokenization function
def tokenize_function(examples):
    result = tokenizer(examples["comment_text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

# Use batched=True to activate fast multithreading!
tokenized_datasets = train_dataset.map(tokenize_function, batched=True, remove_columns=["address", "stance", "date", "comment_text"])

tokenized_datasets

Map:   0%|          | 0/542 [00:00<?, ? examples/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Token indices sequence length is longer than the specified maximum sequence length for this model (2536 > 512). Running this sequence through the model will result in indexing errors
Map: 100%|██████████| 542/542 [00:00<00:00, 5085.23 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'word_ids'],
    num_rows: 542
})

In [10]:
print(f'Tokenizer max length: {tokenizer.model_max_length}')

Tokenizer max length: 512


In [11]:
# Get a sense of the number of tokens in each sample
for idx, sample in enumerate(tokenized_datasets[:3]["input_ids"]):
    print(f"Sample {idx} has {len(sample)} tokens.")

Sample 0 has 12 tokens.
Sample 1 has 2536 tokens.
Sample 2 has 14 tokens.


Concatenate and chunk the data according to the maximum chunk size. 

In [12]:
# Set maximum chunk size 
chunk_size=128

def group_texts(examples):
    # Concatenate all the comment texts 
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts 
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len 
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [13]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)

Map: 100%|██████████| 542/542 [00:00<00:00, 1336.24 examples/s]


In [14]:
tokenizer.decode(lm_datasets[0]["input_ids"])

'[CLS] the new development will increase noise pollution in this area [SEP] [CLS] the proposed development is contrary to national, regional or local planning policy, as the development fails to satisfy the minimum requirement of having a neutral impact on air pollution. the proposed development contradicts the national clean air policy, along with clean air policies by both the mayor of london and mayor of newham. the proposed development is not in keeping with the stylistic context or scale of the local area. the proposed development is a blot on the local area. considerable efforts have been made by neighbouring residential developments to add stylistic context to this historically significant area. the bulldozing of'

Mask some of the tokens for training.

In [15]:
# mlm is the fraction of tokens to mask - 15% is popular in the literature. 
mlm_prob = 0.15

# mask the tokens
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=mlm_prob)

Soem examples of what this type of masking looks like.

In [16]:
samples = [lm_datasets[i] for i in range(2)]
for sample in samples:
    _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
    print(tokenizer.decode(chunk))

[CLS] the new development will [MASK] noise pollution in this area [SEP] [CLS] the proposed development is contrary to national [MASK] regional or local planning policy, as the development fails [MASK] satisfy the [MASK] requirement of [MASK] a neutral impact on [MASK] pollution. the proposed development contradicts the national clean air policy linnaeus along with clean air policies by both the provide [MASK] london and mayor of [MASK]ham. the proposed development is not50 keeping with the stylistic context or scale of the [MASK] area. the generous development is a [MASK]lot on the local area. [MASK] [MASK] have been made by neighbouring residential developments to add [MASK] context [MASK] this historically significant area. the bulldozing of
[MASK] the last remaining marina in east london to make way for [MASK] industrial [MASK] is out of scale with the local neighbourhood. in addition, a multi - storey distribution centre is outside of the stylistic context of [MASK] local area and

Alternative function for masking - this masks whole words, rather than masking tokens (which could be parts of words). 

In [17]:
wwm_probability = 0.2

def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # create a map between words and corresponding token indices 
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)

        
        # randomly mask whole words 
        mask = np.random.binomial(1, wwm_probability, (len(mapping)))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels

    return default_data_collator(features)

Examples of what this type of masking looks like.

In [18]:
samples = [lm_datasets[i] for i in range(2)]
batch = whole_word_masking_data_collator(samples)

for chunk in batch["input_ids"]:
    print(tokenizer.decode(chunk))

[CLS] the new development will increase noise pollution in this area [SEP] [CLS] the [MASK] development is contrary to [MASK], [MASK] [MASK] local planning [MASK] [MASK] as the development fails [MASK] satisfy the minimum requirement of having [MASK] neutral impact on air pollution. the proposed development contradicts [MASK] national clean air policy, along [MASK] clean air policies by [MASK] [MASK] mayor of london and [MASK] of newham. the proposed [MASK] is not [MASK] keeping [MASK] the stylistic context or scale of the local [MASK]. the proposed development is a blot on the local area. considerable [MASK] have [MASK] [MASK] by neighbouring residential developments to add stylistic context to [MASK] historically significant [MASK]. the bulldozing of
probably the last remaining marina in east london to make way [MASK] an industrial shipyard is out of [MASK] with the local neighbourhood. [MASK] addition, a multi [MASK] storey distribution centre [MASK] [MASK] [MASK] the stylistic [MAS

## Downsample the data 
NOTE: I've accidentally split the data twice - since I already had a train/test split when I loaded the datasets into the nb. 

In [19]:
downsampled_dataset = lm_datasets.train_test_split(test_size=0.2)

In [21]:
len(downsampled_dataset["train"])

793

## Train the model!!!

In [48]:
batch_size = 8

logging_steps = len(downsampled_dataset["train"]) // batch_size
training_args = TrainingArguments(
    output_dir="../outputs",
    overwrite_output_dir=True,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    logging_steps=logging_steps,
    fp16=False,
    bf16=True # Note this enables bfloat16 conversion which is supported by the Apple MPD backend
)

In [49]:
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer 
)

/var/folders/4n/x6w1yfcx01qbymrsfpz4ybq00000gn/T/ipykernel_15868/1257576995.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [50]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 28.38


In [29]:
# run the training loop!
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,2.806200,2.525695,0.001800
2,2.587200,2.516653,0.001800
3,2.479100,2.450331,0.001800


TrainOutput(global_step=300, training_loss=2.6175007804234824, metrics={'train_runtime': 170.1777, 'train_samples_per_second': 13.98, 'train_steps_per_second': 1.763, 'total_flos': 78840747588096.0, 'train_loss': 2.6175007804234824, 'epoch': 3.0})

## Model performance 

In [30]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 11.61


See how the model performs on some MASKed sentences.

In [31]:
mask_filler = pipeline("fill-mask", model=model, tokenizer=tokenizer)

preds = mask_filler(text)
for pred in preds:
    print(f"{pred['sequence']}")

Device set to use mps:0


this is an awful sight.
this is an awful place.
this is an awful idea.
this is an awful thing.
this is an awful experience.


In [32]:
text1 = "I [MASK] this planning [MASK]"
mask_filler = pipeline("fill-mask", model=model, tokenizer=tokenizer)


while "[MASK]" in text1:
    preds = mask_filler(text1)  # Get predictions for the first [MASK]
    
    if isinstance(preds, list) and isinstance(preds[0], list):  # Check if multiple masks were processed at once
        preds = preds[0]  # Take only the first mask's predictions

    best_pred = preds[0]["token_str"]  # Choose the top predicted word
    text1 = text1.replace("[MASK]", best_pred, 1)  # Replace only the first [MASK]

print(text1)

Device set to use mps:0


I need this planning .


In [33]:
preds

[{'score': 0.7391762733459473,
  'token': 1012,
  'token_str': '.',
  'sequence': 'i need this planning.'},
 {'score': 0.04452500864863396,
  'token': 2832,
  'token_str': 'process',
  'sequence': 'i need this planning process'},
 {'score': 0.02115713432431221,
  'token': 6912,
  'token_str': 'exercise',
  'sequence': 'i need this planning exercise'},
 {'score': 0.01771572418510914,
  'token': 6994,
  'token_str': 'tool',
  'sequence': 'i need this planning tool'},
 {'score': 0.017516275867819786,
  'token': 1025,
  'token_str': ';',
  'sequence': 'i need this planning ;'}]

## Save the model 

This saves the models weights and parameters so it can be used for downstream tasks. 

In [34]:
model.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")
tokenizer.save_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")

('../outputs/nlp_fine_tuning/distilbert-base-uncased/tokenizer_config.json',
 '../outputs/nlp_fine_tuning/distilbert-base-uncased/special_tokens_map.json',
 '../outputs/nlp_fine_tuning/distilbert-base-uncased/vocab.txt',
 '../outputs/nlp_fine_tuning/distilbert-base-uncased/added_tokens.json',
 '../outputs/nlp_fine_tuning/distilbert-base-uncased/tokenizer.json')